# Mamba SCA — Training & Evaluation

**Paper**: "Deep Learning-based Side-Channel Attack: Mamba Approach"  
**Author**: Beatrise Bertule, Radboud University Nijmegen, January 2026

> All model code and training utilities live in the `.py` files of the repo.  
> Edit those files to make changes — this notebook just orchestrates them.

---

## Cell 1 · Clone Repo & Install Dependencies

In [ ]:
import os, sys

REPO_URL  = "https://github.com/loshithan/Beatrise-B--ertule_mamba_sca.git"
REPO_DIR  = "Beatrise-B--ertule_mamba_sca"   # local folder name after clone

# ── Clone (skip if already present) ───────────────────────────────────
if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL} …")
    ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError("git clone failed — check the URL and your internet connection.")
    print("Clone complete.")
else:
    print(f"Repo already present at ./{REPO_DIR} — pulling latest …")
    os.system(f"git -C {REPO_DIR} pull")

# ── Add repo root to Python path so we can import the .py modules ─────
REPO_ABS = os.path.abspath(REPO_DIR)
if REPO_ABS not in sys.path:
    sys.path.insert(0, REPO_ABS)
print(f"sys.path includes: {REPO_ABS}")

# ── Dataset path (ASCAD.h5 must be in repo/data/ or set manually) ─────
DATA_PATH = os.path.join(REPO_ABS, "data", "ASCAD.h5")
if not os.path.isfile(DATA_PATH):
    # Colab/alternative: set DATA_PATH manually if dataset is stored elsewhere
    DATA_PATH = "/content/ASCAD.h5"   # <-- change this if needed
print(f"Dataset path : {DATA_PATH}")
print(f"Dataset found: {os.path.isfile(DATA_PATH)}")

## Cell 2 · Imports from Repo `.py` Files

In [ ]:
# ── Standard library ───────────────────────────────────────────────────
import math, time
import numpy as np
import h5py
import matplotlib.pyplot as plt
import torch

# ── Repo imports (all logic lives here — edit the .py files to change behaviour) ──
from mamba_sca_model import MambaSCAModel          # architecture
from train import (                                 # training utilities
    make_labels,
    compute_guessing_entropy,
    train,
    evaluate_attack,
    AES_SBOX,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 3 · Configuration

All hyperparameters match the paper exactly (Tables 4.3, 4.4).  
Change these here — they are passed into `train()` and `evaluate_attack()` from `train.py`.

In [ ]:
# ── Data ───────────────────────────────────────────────────────────────
TARGET_BYTE = 2          # ASCADv1 target byte (Section 4.4.1)
N_TRAIN     = 40_000     # Table 4.3
N_VAL       = 10_000     # Table 4.3

# ── Model (Table 4.2) ──────────────────────────────────────────────────
TRACE_LENGTH = 700       # ASCADv1 pre-selected POIs
D_MODEL      = 64
N_BLOCKS     = 2
NUM_CLASSES  = 256
D_STATE      = 16         # gives ~148k params, closest to paper
D_CONV       = 4
EXPAND       = 2

# ── Training (Table 4.4 / Section 4.4.2) ──────────────────────────────
SAVE_PATH    = "best_model.pt"

print("Configuration:")
print(f"  TARGET_BYTE  = {TARGET_BYTE}")
print(f"  N_TRAIN      = {N_TRAIN:,}")
print(f"  N_VAL        = {N_VAL:,}")
print(f"  D_MODEL      = {D_MODEL}    D_STATE = {D_STATE}")
print(f"  N_BLOCKS     = {N_BLOCKS}      DEVICE  = {DEVICE}")

## Cell 4 · Load ASCADv1 Dataset

In [ ]:
print(f"Loading: {DATA_PATH}")

with h5py.File(DATA_PATH, "r") as f:
    # Print HDF5 layout
    print("\nHDF5 structure:")
    def _show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name:55s} shape={obj.shape}  dtype={obj.dtype}")
    f.visititems(_show)
    print()

    # Profiling (train + val)
    X_profiling  = np.array(f["Profiling_traces/traces"],             dtype=np.float32)
    pt_profiling = np.array(f["Profiling_traces/metadata"]["plaintext"])
    key_profile  = np.array(f["Profiling_traces/metadata"]["key"])

    # Attack (test)
    X_attack     = np.array(f["Attack_traces/traces"],                dtype=np.float32)
    pt_attack    = np.array(f["Attack_traces/metadata"]["plaintext"])
    key_attack   = np.array(f["Attack_traces/metadata"]["key"])

correct_key_byte = int(key_profile[0][TARGET_BYTE])
print(f"Profiling traces : {X_profiling.shape}")
print(f"Attack traces    : {X_attack.shape}")
print(f"Trace length     : {X_profiling.shape[1]}")
print(f"Correct key byte : 0x{correct_key_byte:02X}  ({correct_key_byte})")

## Cell 5 · Split, Label & Normalise

In [ ]:
# ── Splits (Table 4.3) ─────────────────────────────────────────────────
X_train  = X_profiling[:N_TRAIN]
X_val    = X_profiling[N_TRAIN : N_TRAIN + N_VAL]
X_test   = X_attack

pt_train = pt_profiling[:N_TRAIN]
pt_val   = pt_profiling[N_TRAIN : N_TRAIN + N_VAL]
pt_test  = pt_attack

# ── Labels — uses make_labels() from train.py ──────────────────────────
# make_labels(plaintexts, key, target_byte) → S-box[pt XOR k]  (Eq 4.1/4.2)
y_train = make_labels(pt_train, key_profile[0],  TARGET_BYTE)
y_val   = make_labels(pt_val,   key_profile[N_TRAIN], TARGET_BYTE)

print(f"Train : {X_train.shape}  labels [{y_train.min()}, {y_train.max()}]")
print(f"Val   : {X_val.shape}  labels [{y_val.min()}, {y_val.max()}]")
print(f"Test  : {X_test.shape}  (attack evaluation only)")

# Quick class-balance check
uq, cts = np.unique(y_train, return_counts=True)
print(f"\nLabel distribution: {len(uq)} classes, "
      f"min={cts.min()}, max={cts.max()}, mean≈{cts.mean():.0f}")

## Cell 6 · Build the Model

In [ ]:
# MambaSCAModel is defined in mamba_sca_model.py
# Edit that file to change the architecture; re-run this cell to rebuild.
model = MambaSCAModel(
    trace_length = X_train.shape[1],
    d_model      = D_MODEL,
    n_blocks     = N_BLOCKS,
    num_classes  = NUM_CLASSES,
    d_state      = D_STATE,
    d_conv       = D_CONV,
    expand       = EXPAND,
)

n_params = model.count_parameters()
print(f"Model         : MambaSCAModel  (from mamba_sca_model.py)")
print(f"Trace length  : {X_train.shape[1]}")
print(f"Parameters    : {n_params:,}  (paper reports ~148,000)")
match = '✅ YES' if 140_000 <= n_params <= 160_000 else '⚠️ CLOSE' if 100_000 <= n_params <= 200_000 else '❌ NO'
print(f"Param match   : {match}")
print(f"\n{model}")

## Cell 7 · Train

`train()` is imported directly from **`train.py`**.  
Printing per step and per epoch is handled inside that function.

**What you see per epoch:**
- Every 10 mini-batch steps → `batch_loss` + `running_avg`  
- End of epoch → `train_loss`, `val_loss`, `val_GE`, trend deltas  
- `★ NEW BEST` line when a checkpoint is saved

In [ ]:
# train() signature (from train.py):
#   train(model, X_train, y_train, X_val, y_val,
#         pt_val, correct_key_val, target_byte,
#         save_path, device)
#
# Hyperparameters (batch_size=768, lr=5e-3, weight_decay=1e-4,
# max_epochs=100, GE validation) are set inside train.py.

t0 = time.time()

best_ge, best_epoch = train(
    model           = model,
    X_train         = X_train,
    y_train         = y_train,
    X_val           = X_val,
    y_val           = y_val,
    pt_val          = pt_val,
    correct_key_val = correct_key_byte,
    target_byte     = TARGET_BYTE,
    save_path       = SAVE_PATH,
    device          = DEVICE,
)

elapsed = time.time() - t0
print(f"\nTotal training time : {elapsed/60:.1f} min")
print(f"Best val GE         : {best_ge:.4f}  at epoch {best_epoch}")
print(f"Checkpoint saved to : {SAVE_PATH}")

## Cell 8 · Load Best Checkpoint

In [ ]:
ckpt = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)
model.eval()

ckpt_mean, ckpt_std = ckpt["mean_std"]
print(f"Loaded checkpoint from epoch {ckpt['epoch']}  (val_GE = {ckpt['val_ge']:.4f})")

## Cell 9 · Attack Evaluation

`evaluate_attack()` is imported directly from **`train.py`**  
and computes key rank, GE, and convergence curve (Section 4.4.3).

In [ ]:
# evaluate_attack() signature (from train.py):
#   evaluate_attack(model, X_test, pt_test, correct_key,
#                   target_byte, mean, std, device, eps)
#
# Returns dict with:
#   key_rank, mean_ge, median_ge, ge_convergence,
#   ge_conv_value, ge_conv_trace

results = evaluate_attack(
    model       = model,
    X_test      = X_test,
    pt_test     = pt_test,
    correct_key = correct_key_byte,
    target_byte = TARGET_BYTE,
    mean        = ckpt_mean,
    std         = ckpt_std,
    device      = DEVICE,
)

key_rank       = results["key_rank"]
mean_ge        = results["mean_ge"]
median_ge      = results["median_ge"]
ge_convergence = results["ge_convergence"]
ge_conv_value  = results["ge_conv_value"]
ge_conv_trace  = results["ge_conv_trace"]

if key_rank == 0:
    print("\n🎉 Key rank = 0 — correct key is the TOP candidate!")
elif key_rank <= 5:
    print(f"\n✅ Key rank = {key_rank} — practically broken.")
else:
    print(f"\n⚠️  Key rank = {key_rank} — may need more training.")

## Cell 10 · GE Convergence Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── GE convergence curve ───────────────────────────────────────────────
axes[0].plot(range(1, 1001), ge_convergence, "b-", linewidth=1.5, label="Mean GE")
axes[0].axhline(y=1.0, color="red",   linestyle="--", alpha=0.6, label="GE = 1")
axes[0].axhline(y=0.0, color="green", linestyle="--", alpha=0.6, label="GE = 0 (perfect)")
if ge_conv_trace < 1000:
    axes[0].axvline(x=ge_conv_trace, color="orange", linestyle="--", alpha=0.7,
                    label=f"GE ≤ 1 at trace {ge_conv_trace}")
axes[0].set_xlabel("Number of Attack Traces")
axes[0].set_ylabel("Mean Guessing Entropy")
axes[0].set_title("GE Convergence (100 simulations avg)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 1000)

# ── Key rank summary ───────────────────────────────────────────────────
labels_s  = ["Key Rank\n(10k traces)", "Mean GE\n(100 runs)", "Median GE\n(100 runs)"]
values_s  = [key_rank, mean_ge, median_ge]
colors_s  = ["steelblue", "darkorange", "seagreen"]
bars = axes[1].bar(labels_s, values_s, color=colors_s, width=0.5)
for bar, val in zip(bars, values_s):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{val:.2f}", ha="center", va="bottom", fontweight="bold")
axes[1].axhline(y=1.0, color="red", linestyle="--", alpha=0.6, label="GE = 1 threshold")
axes[1].set_ylabel("Value")
axes[1].set_title("Attack Evaluation Summary")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("evaluation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: evaluation_results.png")

## Cell 11 · Final Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                      EXPERIMENT SUMMARY                         ║
╠══════════════════════════════════════════════════════════════════╣""")
print(f"║  Repo         : {REPO_URL[:50]:<50s}  ║")
print(f"║  Model        : MambaSCAModel  ({n_params:,} params){'':>15s}║")
print(f"║  Dataset      : ASCADv1  (target byte {TARGET_BYTE}){'':>28s}║")
print(f"║  Device       : {DEVICE:<50s}  ║")
print( "╠══════════════════════════════════════════════════════════════════╣")
print(f"║  Best epoch   : {best_epoch:<52d}║")
print(f"║  Best val GE  : {best_ge:<52.4f}║")
print( "╠══════════════════════════════════════════════════════════════════╣")
print(f"║  Key rank (10k traces)  : {key_rank:<42d}║")
print(f"║  Mean GE   (100 runs)   : {mean_ge:<42.4f}║")
print(f"║  Median GE (100 runs)   : {median_ge:<42.4f}║")
print(f"║  Traces to GE ≤ 1       : {ge_conv_trace:<42d}║")
print( "╚══════════════════════════════════════════════════════════════════╝")